Building an End-to-End Data Engineering Pipeline for E-Commerce Order Analytics

Step 1: EXTRACT - Baca Data Mentah

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np  
import ast
# Import semua data raw 
data = pd.read_csv("../data/raw/raw_suppliers.csv")

data.head(11)

,supplier_id,product_id,nama_supplier,kontak,kota,lead_time_hari
0,SUP-001,P001,Gudang Kosmetik Utama,021-4749746,Cikarang,-1.0
1,SUP-002,P002,Toko Bahan Baku Kosmetik,NaN,bandung,7.0
2,SUP-003,P003,Toko Bahan Baku Kosmetik,021-2455921,Tangerang,7.0
3,SUP-004,P004,PT Kemasan Halal Indonesia,021-5548518,Tangerang,NaN
4,SUP-005,P005,CV Packaging Solusi,021-1807451,Cikarang,3.0
5,SUP-006,P006,Toko Bahan Baku Kosmetik,021-3651911,Bandung,3.0
6,SUP-007,P007,CV Botol & Tube Indonesia,021-9966798,bandung,2.0
7,SUP-008,P008,PT Fragrance Import Indonesia,021-3079429,Bandung,NaN
8,SUP-009,P009,Toko Bahan Baku Kosmetik,021-2425848,Bandung,3.0
9,SUP-010,P010,PT Distribusi Personal Care,021-5106213,Bandung,7.0


In [2]:
# Inspeksi awal
print(f"Jumlah baris: {len(data)}")
print(f"Kolom: {list(data.columns)}")
data.info()

Jumlah baris: 11
Kolom: ['supplier_id', 'product_id', 'nama_supplier', 'kontak', 'kota', 'lead_time_hari']
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   supplier_id     11 non-null     str    
 1   product_id      11 non-null     str    
 2   nama_supplier   11 non-null     str    
 3   kontak          10 non-null     str    
 4   kota            11 non-null     str    
 5   lead_time_hari  9 non-null      float64
dtypes: float64(1), str(5)
memory usage: 660.0 bytes


In [3]:
print("\nDuplikasi")
print(f"{data.duplicated().sum()}")


Duplikasi
0


In [4]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
supplier_id       0
product_id        0
nama_supplier     0
kontak            1
kota              0
lead_time_hari    2
dtype: int64


In [5]:
print(f"\nHarga negatif: {(data['lead_time_hari'] < 0).sum()}")


Harga negatif: 1


Step 2: TRANSFORM - Bersihkan Data

In [6]:
# Kontak Missing Value
# Menambahakan Kontak pada kolom kontak  
data['kontak'] = data['kontak'].fillna('021-2455342')
print(data['kontak'])

0     021-4749746
1     021-2455342
2     021-2455921
3     021-5548518
4     021-1807451
5     021-3651911
6     021-9966798
7     021-3079429
8     021-2425848
9     021-5106213
10    021-2455921
Name: kontak, dtype: str


In [7]:
# Menyamakaan penulisan kota 
data['kota'] = (data['kota'].fillna('').str.strip().str.title())
print(data['kota'])

0      Cikarang
1       Bandung
2     Tangerang
3     Tangerang
4      Cikarang
5       Bandung
6       Bandung
7       Bandung
8       Bandung
9       Bandung
10    Tangerang
Name: kota, dtype: str


In [8]:
# Mengubah nilai negatif pada lead_time_hari pada kota Cikarang menjadi Nan
# Ubah nilai negatif menjadi NaN
data.loc[data['lead_time_hari'] < 0, 'lead_time_hari'] = np.nan

In [9]:
print(data[['kota', 'lead_time_hari']])

         kota  lead_time_hari
0    Cikarang             NaN
1     Bandung             7.0
2   Tangerang             7.0
3   Tangerang             NaN
4    Cikarang             3.0
5     Bandung             3.0
6     Bandung             2.0
7     Bandung             NaN
8     Bandung             3.0
9     Bandung             7.0
10  Tangerang             7.0


In [10]:
# Untuk mengisi missing value pada kolom lead_time_hari 
# Menggunkan rata-rata atau mean dari kota yang sama dari Missing Value-nya
# Menghitung rata-rata lead time per kota
mean = (data.groupby('kota')['lead_time_hari'].mean().round(2))
print(mean)

kota
Bandung      4.4
Cikarang     3.0
Tangerang    7.0
Name: lead_time_hari, dtype: float64


In [11]:
# Mengisi missing value/Nan dari lead_time_hari berdasarakan mean
# dari kota masing-masing
data.loc[(data['kota'] == 'Bandung') &(data['lead_time_hari'].isna()),'lead_time_hari'] = 4.0
data.loc[(data['kota'] == 'Cikarang') &(data['lead_time_hari'].isna()),'lead_time_hari'] = 3.0
data.loc[(data['kota'] == 'Tangerang') &(data['lead_time_hari'].isna()),'lead_time_hari'] = 7.0
# otomatis
# mean = data.groupby('kota')['lead_time_hari'].transform('mean')
# data['lead_time_hari'] = data['lead_time_hari'].fillna(mean)

In [12]:
print(data[['kota', 'lead_time_hari']])

         kota  lead_time_hari
0    Cikarang             3.0
1     Bandung             7.0
2   Tangerang             7.0
3   Tangerang             7.0
4    Cikarang             3.0
5     Bandung             3.0
6     Bandung             2.0
7     Bandung             4.0
8     Bandung             3.0
9     Bandung             7.0
10  Tangerang             7.0


In [13]:
# Pada kolom lead_time_hari 
# Akan akan di ubah menjadi int/number biasa
data['lead_time_hari'].isna().sum()
data['lead_time_hari'] = data['lead_time_hari'].astype(int)

Memastiakn Apakah Data Sudah bersih dan Sesuai Standar

In [14]:
# Cek data
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   supplier_id     11 non-null     str  
 1   product_id      11 non-null     str  
 2   nama_supplier   11 non-null     str  
 3   kontak          11 non-null     str  
 4   kota            11 non-null     str  
 5   lead_time_hari  11 non-null     int64
dtypes: int64(1), str(5)
memory usage: 660.0 bytes


In [15]:
# Simpan ke CSV data setelah proses atau clean
data.to_csv(
    "../data/warehouse/suppliers_clean.csv",
    index=False,
    encoding="utf-8")
# Melihat kembali data yang disimpan
data = pd.read_csv("../data/warehouse/suppliers_clean.csv")
data.head(12)

,supplier_id,product_id,nama_supplier,kontak,kota,lead_time_hari
0,SUP-001,P001,Gudang Kosmetik Utama,021-4749746,Cikarang,3
1,SUP-002,P002,Toko Bahan Baku Kosmetik,021-2455342,Bandung,7
2,SUP-003,P003,Toko Bahan Baku Kosmetik,021-2455921,Tangerang,7
3,SUP-004,P004,PT Kemasan Halal Indonesia,021-5548518,Tangerang,7
4,SUP-005,P005,CV Packaging Solusi,021-1807451,Cikarang,3
5,SUP-006,P006,Toko Bahan Baku Kosmetik,021-3651911,Bandung,3
6,SUP-007,P007,CV Botol & Tube Indonesia,021-9966798,Bandung,2
7,SUP-008,P008,PT Fragrance Import Indonesia,021-3079429,Bandung,4
8,SUP-009,P009,Toko Bahan Baku Kosmetik,021-2425848,Bandung,3
9,SUP-010,P010,PT Distribusi Personal Care,021-5106213,Bandung,7
